In [1]:
# ==============================================================================
# LC ETHNOGRAPHIC TERMS (AFSET) INGESTION: STEP 1
# 
# ARCHITECTURE NOTE: AFSET is a standard W3C Semantic Web vocabulary built on SKOS. 
# Instead of crawling a live API and hitting rate limits, this script uses a Local 
# Bulk Parsing strategy. It loads the entire N-Triples (.nt) graph into memory.
#
# SSSOM ALIGNMENT STRATEGY: 
# The script extracts strict SKOS hierarchical relationships (broader/narrower) 
# to build the vertical tree. Internal lateral links (skos:related) are 
# intentionally ignored here and reserved for Step 2 Lateral Discovery to 
# keep the core SSSOM taxonomy mathematically pure.
# ==============================================================================
# INSTRUCTIONS FOR REPRODUCIBILITY
# 1. Navigate to the Library of Congress Bulk Download page: https://id.loc.gov/download/
# 2. Locate the "Ethnographic Thesaurus" dataset.
# 3. Download the "SKOS/RDF N-Triples" format file (ethnographicTerms.skosrdf.nt.gz).
# 4. Extract the .gz archive and place `ethnographicTerms.skosrdf.nt` in your 
#    local `data/external/` directory.
# ==============================================================================

import os
import sys
import pandas as pd
import time
from dotenv import load_dotenv

try:
    from rdflib import Graph, URIRef, Namespace
    from rdflib.namespace import RDF, SKOS
except ImportError:
    print("CRITICAL: rdflib is required for bulk processing. Run: pip install rdflib")
    sys.exit(1)

# --- 1. Seed List ---
SEED_URIS = {
    # --- Original High-Level Seeds ---
    "belief": "http://id.loc.gov/vocabulary/ethnographicTerms/afset001590",
    "religious education": "http://id.loc.gov/vocabulary/ethnographicTerms/afset015192",
    "sacred": "http://id.loc.gov/vocabulary/ethnographicTerms/afset015699",
    "ritual": "http://id.loc.gov/vocabulary/ethnographicTerms/afset015441",
    "otherworlds": "http://id.loc.gov/vocabulary/ethnographicTerms/afset012970",
    "supernatural time": "http://id.loc.gov/vocabulary/ethnographicTerms/afset017959",
    "paranormal communication": "http://id.loc.gov/vocabulary/ethnographicTerms/afset013194",
    "religious art": "http://id.loc.gov/vocabulary/ethnographicTerms/afset015182",
    "supernatural beings": "http://id.loc.gov/vocabulary/ethnographicTerms/afset017950",
    
    # --- New Seeds from Lateral Discovery ---
    "ascetics": "http://id.loc.gov/vocabulary/ethnographicTerms/afset000935",
    "religious leaders": "http://id.loc.gov/vocabulary/ethnographicTerms/afset015199",
    "pilgrims": "http://id.loc.gov/vocabulary/ethnographicTerms/afset013745",
    "mahatmas": "http://id.loc.gov/vocabulary/ethnographicTerms/afset010946",
    "prayer books": "http://id.loc.gov/vocabulary/ethnographicTerms/afset014295",
    "religious festivals": "http://id.loc.gov/vocabulary/ethnographicTerms/afset015193",
    "religious life": "http://id.loc.gov/vocabulary/ethnographicTerms/afset021017",
    "spiritual life": "http://id.loc.gov/vocabulary/ethnographicTerms/afset021016",
    "purity (ritual)": "http://id.loc.gov/vocabulary/ethnographicTerms/afset014721",
    "crimes against religion": "http://id.loc.gov/vocabulary/ethnographicTerms/afset015716",
    "caliphs": "http://id.loc.gov/vocabulary/ethnographicTerms/afset002560",
    "religious buildings": "http://id.loc.gov/vocabulary/ethnographicTerms/afset015183",
    "haunted houses": "http://id.loc.gov/vocabulary/ethnographicTerms/afset008510",
    "ritual clothing": "http://id.loc.gov/vocabulary/ethnographicTerms/afset015443",
    "divination equipment": "http://id.loc.gov/vocabulary/ethnographicTerms/afset005334",
    "religious songs": "http://id.loc.gov/vocabulary/ethnographicTerms/afset015211",
    "sacred music": "http://id.loc.gov/vocabulary/ethnographicTerms/afset015706",
    "cantors": "http://id.loc.gov/vocabulary/ethnographicTerms/afset002678",
    "muezzins": "http://id.loc.gov/vocabulary/ethnographicTerms/afset012039",
    "religious institutions": "http://id.loc.gov/vocabulary/ethnographicTerms/afset015197",
    "haunted places": "http://id.loc.gov/vocabulary/ethnographicTerms/afset008511",
    "grace at meals": "http://id.loc.gov/vocabulary/ethnographicTerms/afset008047",

    # --- Seeds from Lateral Discovery round 2---
    "religious drama": "http://id.loc.gov/vocabulary/ethnographicTerms/afset015191",
    "prophets": "http://id.loc.gov/vocabulary/ethnographicTerms/afset014531",
    "desecration": "http://id.loc.gov/vocabulary/ethnographicTerms/afset005050",
    "sacred vocal music": "http://id.loc.gov/vocabulary/ethnographicTerms/afset021194"
}

# --- 2. Environment & Schema Setup ---
load_dotenv("../../config/.env")

current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
config_dir = os.path.abspath(os.path.join(current_dir, "..", "..", "config"))
raw_data_dir = os.path.abspath(os.path.join(current_dir, "..", "..", "data", "raw"))
external_data_dir = os.path.abspath(os.path.join(current_dir, "..", "..", "data", "external"))

os.makedirs(raw_data_dir, exist_ok=True)
os.makedirs(external_data_dir, exist_ok=True)

sys.path.append(config_dir)
try:
    from ingest_schema_manager import get_registry_info, finalize_row, COLUMN_ORDER
except ImportError:
    print("CRITICAL: ingest_schema_manager.py not found. Check PYTHONPATH.")
    sys.exit(1)

SOURCE_PREFIX = "AFSET"
registry_data = get_registry_info(prefix=SOURCE_PREFIX, config_dir=config_dir)

raw_ingest_file = os.path.join(raw_data_dir, "raw_afset_bulk.csv")
bulk_source_file = os.path.join(external_data_dir, "ethnographicTerms.skosrdf.nt") 

BASE_URI = "http://id.loc.gov/vocabulary/ethnographicTerms"
MADS = Namespace("http://www.loc.gov/mads/rdf/v1#")

# --- 3. Local Validation & Caching ---
if not os.path.exists(bulk_source_file):
    print(f"CRITICAL: Bulk source file missing at {bulk_source_file}.")
    print("Please follow the reproducibility instructions at the top of this script.")
    sys.exit(1)

if os.path.exists(raw_ingest_file):
    existing_df = pd.read_csv(raw_ingest_file, dtype={'Concept_ID': str})
    if list(existing_df.columns) != COLUMN_ORDER:
        print(f"[!] Schema mismatch detected. Deleting outdated {os.path.basename(raw_ingest_file)}...")
        os.remove(raw_ingest_file)
        processed_ids = set()
    else:
        processed_ids = set(existing_df['Concept_ID'].dropna().tolist())
        print(f"[*] Resuming run. Found {len(processed_ids)} previously processed concepts.")
else:
    processed_ids = set()
    print(f"[*] Starting fresh {SOURCE_PREFIX} bulk ingest.")

# --- 4. Load Graph into Memory ---
print(f"[*] Loading bulk graph file into memory. This may take a moment...")
start_time = time.time()
g = Graph()
g.parse(bulk_source_file, format="nt") 
print(f"[*] Graph loaded in {time.time() - start_time:.2f} seconds. Contains {len(g)} triples.")

# --- 5. Helper Functions ---
def get_english_value(subject_node, predicate):
    """Extracts an English or untagged string value from the rdflib Graph."""
    for obj in g.objects(subject_node, predicate):
        if hasattr(obj, 'language'):
            if obj.language == 'en' or obj.language is None:
                return str(obj)
        else:
            return str(obj)
    return ""

def get_english_values_list(subject_node, predicate):
    """Extracts all English or untagged string values from the rdflib Graph as a list."""
    values = []
    for obj in g.objects(subject_node, predicate):
        if hasattr(obj, 'language'):
            if obj.language == 'en' or obj.language is None:
                values.append(str(obj))
        else:
            values.append(str(obj))
    return list(set(values)) # Deduplicate to ensure clean output

def build_hierarchy_path(subject_node):
    """Climbs the graph upwards to the absolute root to build the complete breadcrumb trail."""
    path_parts = []
    current = subject_node
    visited = set() # Prevent infinite loops in cyclical graphs
    
    while current and current not in visited:
        visited.add(current)
        
        # Get label for current node
        label = get_english_value(current, SKOS.prefLabel) or get_english_value(current, MADS.authoritativeLabel)
        path_parts.append(label if label else str(current).split('/')[-1])
        
        # Find broader parent (prioritize skos:broader, fallback to inverse skos:narrower)
        broaders = list(g.objects(current, SKOS.broader))
        if not broaders:
            broaders = list(g.subjects(SKOS.narrower, current))
            
        if broaders:
            current = broaders[0] # Select the first parent to build a single linear path
        else:
            current = None
            
    path_parts.reverse()
    return " > ".join(path_parts)

def get_parent_ids(subject_node):
    """Extracts all immediate parent IDs to preserve polyhierarchy data."""
    broaders = list(g.objects(subject_node, SKOS.broader))
    if not broaders:
        broaders = list(g.subjects(SKOS.narrower, subject_node))
        
    p_ids = [str(p).split('/')[-1] for p in broaders if str(p).startswith(BASE_URI)]
    return " | ".join(list(set(p_ids)))

# --- 6. Data Extraction Logic ---
def process_node_local(uri_string):
    """Recursively traverses the in-memory graph to map concepts without depth limits."""
    cid = uri_string.split('/')[-1]
    if cid in processed_ids:
        return
        
    print(f"\rProcessing: {cid}{' ' * 40}", end="", flush=True)
    
    subject_node = URIRef(uri_string)
    
    if not (subject_node, None, None) in g:
        print(f"\n[!] Node {uri_string} not found in the bulk file.")
        return

    # Field Extraction
    pref_label = get_english_value(subject_node, SKOS.prefLabel) or get_english_value(subject_node, MADS.authoritativeLabel)
    scope_note = get_english_value(subject_node, SKOS.scopeNote)
    concept_type = "skos:Concept" if (subject_node, RDF.type, SKOS.Concept) in g else ""
    
    # Synonym Extraction
    synonyms_list = get_english_values_list(subject_node, SKOS.altLabel)
    if not synonyms_list:
        synonyms_list = get_english_values_list(subject_node, MADS.variantLabel)
    synonyms_str = " | ".join(synonyms_list)

    # Map to schema
    extracted_data = {
        "Source_System": "AFS Ethnographic Thesaurus",
        "Primary_Label": pref_label,
        "CURIE": f"{SOURCE_PREFIX}:{cid}",
        "Formal_Label": "",
        "Concept_Type": concept_type,
        "Hierarchy_Path": build_hierarchy_path(subject_node),
        "Synonyms": synonyms_str, 
        "Description": scope_note,
        "Parent_IDs": get_parent_ids(subject_node),
        "Concept_ID": cid,
        "URI": uri_string,
        "Status": "active"
    }

    clean_row = finalize_row(extracted_data)
    pd.DataFrame([clean_row])[COLUMN_ORDER].to_csv(
        raw_ingest_file, mode='a', index=False, 
        header=not os.path.isfile(raw_ingest_file), encoding='utf-8-sig'
    )
    processed_ids.add(cid)

    # Recursive Child Crawl (Unbounded)
    for narrower_node in g.objects(subject_node, SKOS.narrower):
        child_uri = str(narrower_node)
        if child_uri.startswith(BASE_URI):
            process_node_local(uri_string=child_uri)

# --- 7. Main Execution ---
print(f"\nStarting unbounded AFS Ethnographic Thesaurus extraction.")

for label, seed_uri in SEED_URIS.items():
    print(f"\n--- Initiating Seed Branch: {label} ---")
    process_node_local(seed_uri)

print(f"\n\nExtraction complete. Data appended to {raw_ingest_file}")
print(f"Total unique concepts captured: {len(processed_ids)}")

[*] Starting fresh AFSET bulk ingest.
[*] Loading bulk graph file into memory. This may take a moment...
[*] Graph loaded in 15.57 seconds. Contains 511726 triples.

Starting unbounded AFS Ethnographic Thesaurus extraction.

--- Initiating Seed Branch: belief ---
Processing: afset017298                                        
--- Initiating Seed Branch: religious education ---
Processing: afset015192                                        
--- Initiating Seed Branch: sacred ---
Processing: afset015699                                        
--- Initiating Seed Branch: ritual ---
Processing: afset003410                                        
--- Initiating Seed Branch: otherworlds ---
Processing: afset014716                                        
--- Initiating Seed Branch: supernatural time ---
Processing: afset000977                                        
--- Initiating Seed Branch: paranormal communication ---
Processing: afset006436                                        
--- Ini

In [2]:
# ==============================================================================
# CELL 2 / PHASE 2: AFSET LATERAL DISCOVERY (Memory-Only Inbox Approach)
# Run this cell to harvest skos:related links outside the current scope.
# Tags candidates with a 'Discovery_Pass' to keep new items separate from old.
# ==============================================================================

import os
import pandas as pd
import time
from dotenv import load_dotenv

try:
    from rdflib import Graph, URIRef, Namespace
    from rdflib.namespace import RDF, SKOS
except ImportError:
    raise ImportError("CRITICAL: rdflib is required for bulk processing. Run: pip install rdflib")

RUN_LATERAL_DISCOVERY = True  # Toggle to False once ingest script is finalized

if RUN_LATERAL_DISCOVERY:
    # --- 1. Setup & Baseline Load ---
    load_dotenv("../../config/.env")
    current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in locals() else os.getcwd()
    raw_data_dir = os.path.abspath(os.path.join(current_dir, "..", "..", "data", "raw"))
    external_data_dir = os.path.abspath(os.path.join(current_dir, "..", "..", "data", "external"))
    
    raw_afset_file = os.path.join(raw_data_dir, "raw_afset_bulk.csv")
    output_candidates_file = os.path.join(raw_data_dir, "afset_lateral_candidates.csv")
    bulk_source_file = os.path.join(external_data_dir, "ethnographicTerms.skosrdf.nt")
    
    BASE_URI = "http://id.loc.gov/vocabulary/ethnographicTerms"
    MADS = Namespace("http://www.loc.gov/mads/rdf/v1#")
    
    if not os.path.exists(raw_afset_file):
        raise FileNotFoundError(f"Please run Phase 1 to generate the baseline {raw_afset_file} first.")
        
    existing_df = pd.read_csv(raw_afset_file)
    existing_uris = set(existing_df['URI'].astype(str).str.strip().str.rstrip('/'))
    print(f"[*] Loaded {len(existing_uris)} ingested AFSET concepts as the baseline.")

    # --- 2. Build In-Memory Suppression List & Determine Pass Number ---
    seen_uris = set(existing_uris)
    old_candidates_df = pd.DataFrame()
    current_pass = 1
    
    if os.path.exists(output_candidates_file):
        old_candidates_df = pd.read_csv(output_candidates_file)
        
        # Determine the next pass number
        if 'Discovery_Pass' in old_candidates_df.columns:
            current_pass = old_candidates_df['Discovery_Pass'].max() + 1
        else:
            # If an old file exists without the column, backfill it as pass 1
            old_candidates_df['Discovery_Pass'] = 1
            current_pass = 2
            
        if 'Candidate_URI' in old_candidates_df.columns:
            for uri in old_candidates_df['Candidate_URI']:
                seen_uris.add(str(uri).strip())
                
    print(f"[*] Built suppression list of {len(seen_uris)} total URIs (Ingested + Previously Reviewed).")
    print(f"[*] Tagging new discoveries as Pass {current_pass}.")

    # --- 3. Load Graph ---
    print(f"[*] Loading bulk graph file into memory...")
    start_time = time.time()
    g = Graph()
    g.parse(bulk_source_file, format="nt") 
    print(f"[*] Graph loaded in {time.time() - start_time:.2f} seconds.")

    # --- 4. Helper Functions ---
    def get_english_value(subject_node, predicate):
        for obj in g.objects(subject_node, predicate):
            if hasattr(obj, 'language'):
                if obj.language == 'en' or obj.language is None:
                    return str(obj)
            else:
                return str(obj)
        return ""

    def build_hierarchy_path(subject_node):
        path_parts = []
        current = subject_node
        visited = set()
        
        while current and current not in visited:
            visited.add(current)
            label = get_english_value(current, SKOS.prefLabel) or get_english_value(current, MADS.authoritativeLabel)
            path_parts.append(label if label else str(current).split('/')[-1])
            
            broaders = list(g.objects(current, SKOS.broader))
            if not broaders:
                broaders = list(g.subjects(SKOS.narrower, current))
                
            if broaders:
                current = broaders[0]
            else:
                current = None
                
        path_parts.reverse()
        return " > ".join(path_parts)

    # --- 5. Discovery Traversal ---
    candidates_dict = {}
    print(f"\nScanning in-memory graph for lateral SKOS associations...")
    
    for idx, uri_str in enumerate(existing_uris, 1):
        if idx % 50 == 0:
            print(f"\rScanning baseline concept {idx}/{len(existing_uris)}...", end="", flush=True)
            
        subject_node = URIRef(uri_str)
        related_nodes = list(g.objects(subject_node, SKOS.related))
        
        for related_node in related_nodes:
            c_uri = str(related_node)
            
            # Check against the master suppression list
            if c_uri.startswith(BASE_URI) and c_uri not in seen_uris:
                if c_uri not in candidates_dict:
                    c_label = get_english_value(related_node, SKOS.prefLabel) or get_english_value(related_node, MADS.authoritativeLabel)
                    c_path = build_hierarchy_path(related_node)
                    
                    candidates_dict[c_uri] = {
                        'Discovery_Pass': current_pass,
                        'Candidate_URI': c_uri,
                        'Candidate_Label': c_label,
                        'Hierarchy_Path': c_path
                    }
                
                # Capture the parent for context (if the parent hasn't been seen either)
                broaders = list(g.objects(related_node, SKOS.broader))
                if not broaders:
                    broaders = list(g.subjects(SKOS.narrower, related_node))
                    
                if broaders:
                    p_node = broaders[0]
                    p_uri = str(p_node)
                    if p_uri.startswith(BASE_URI) and p_uri not in seen_uris and p_uri not in candidates_dict:
                        p_label = get_english_value(p_node, SKOS.prefLabel) or get_english_value(p_node, MADS.authoritativeLabel)
                        p_path = build_hierarchy_path(p_node)
                        
                        candidates_dict[p_uri] = {
                            'Discovery_Pass': current_pass,
                            'Candidate_URI': p_uri,
                            'Candidate_Label': p_label,
                            'Hierarchy_Path': p_path
                        }

    # --- 6. Export and Tracking ---
    print(f"\n\nLateral Discovery Complete! Found {len(candidates_dict)} NET-NEW candidates.")
    
    if candidates_dict:
        new_candidates_df = pd.DataFrame(list(candidates_dict.values()))
        
        # Combine old candidates with new candidates
        combined_df = pd.concat([old_candidates_df, new_candidates_df], ignore_index=True)
        
        # Sort by Discovery_Pass (Newest First), then by Hierarchy (Alphabetical)
        combined_df = combined_df.sort_values(by=['Discovery_Pass', 'Hierarchy_Path'], ascending=[False, True])
        
        # Ensure column order is clean
        col_order = ['Discovery_Pass', 'Candidate_URI', 'Candidate_Label', 'Hierarchy_Path']
        combined_df = combined_df[col_order]
        
        combined_df.to_csv(output_candidates_file, index=False, encoding='utf-8-sig')
        print(f"Updated {os.path.basename(output_candidates_file)} with the latest candidates for review at the top.")
    else:
        print("No new relevant concepts found on this pass.")
else:
    print("Lateral Discovery is disabled.")

[*] Loaded 997 ingested AFSET concepts as the baseline.
[*] Built suppression list of 1260 total URIs (Ingested + Previously Reviewed).
[*] Tagging new discoveries as Pass 2.
[*] Loading bulk graph file into memory...
[*] Graph loaded in 20.05 seconds.

Scanning in-memory graph for lateral SKOS associations...
Scanning baseline concept 950/997...

Lateral Discovery Complete! Found 0 NET-NEW candidates.
No new relevant concepts found on this pass.
